## 02 - Remove artefacts from 24-h recordings

This notebook detects and masks high-amplitude artefacts in the assembled EDF recordings, using unfiltered signals and interactive threshold adjustment.

Unmasked values and units are preserved (`uV`, `degC`, `a.u.`), except Activity, which is scaled by 10⁻⁶ in the cleaned layer. Missing intervals are documented in JSON sidecars.

**How to use it:** Set `RAT_ID`, `DATASET_ROOT` and `OUTPUT_ROOT` in the settings cell, then run the notebook and review the detected artefacts before saving. The cleaned files are written to `OUTPUT_ROOT/recordings-24h-clean/<RAT_ID>`; `DATASET_ROOT` is only read. The processing log is optional.

## 1 — Configuration

In [ ]:
import json
import mne
import pyedflib
import numpy as np
import calibration   # calibration.py, in the same folder as this notebook
%matplotlib inline
import matplotlib.pyplot as plt
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [ ]:
# ── Settings: edit these ──
RAT_ID = "Rat04"                           # Rat01 ... Rat08
DATASET_ROOT = Path("/path/to/dataset")    # the downloaded dataset; only read
OUTPUT_ROOT = Path("/path/to/output")      # recordings-24h-clean/ is created here

# With OUTPUT_ROOT = DATASET_ROOT, the notebook resumes after the cleaned files that are
# already in the dataset and overwrites the last one of each signal.

In [ ]:
# ── Derived paths ──
RECORDINGS_24H_DIR = DATASET_ROOT / "recordings-24h" / RAT_ID
if not RECORDINGS_24H_DIR.exists():
    raise FileNotFoundError(f"EDF source directory does not exist: {RECORDINGS_24H_DIR}")

if not RECORDINGS_24H_DIR.is_dir():
    raise NotADirectoryError(f"Expected a directory, found: {RECORDINGS_24H_DIR}")

# ── Output folder for cleaned EDFs ──
RECORDINGS_24H_CLEAN_DIR = OUTPUT_ROOT / "recordings-24h-clean" / RAT_ID
CLEAN_DIR_EXISTS = RECORDINGS_24H_CLEAN_DIR.exists()
if CLEAN_DIR_EXISTS and not RECORDINGS_24H_CLEAN_DIR.is_dir():
    raise NotADirectoryError(f"Expected a directory, found: {RECORDINGS_24H_CLEAN_DIR}")
if not CLEAN_DIR_EXISTS:
    RECORDINGS_24H_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# ── Default artefact detection threshold (z-score) ──
DEFAULT_THRESHOLD = 10.0

# ── Extra skip at zero-padded edges (seconds) ──
EDGE_ZERO_SKIP_SEC = 1.05

# CSV file that records one audit row for each successfully saved EDF.
WRITE_PROCESSING_LOG = True  # <── set to False to not write the CSV
PROCESSING_LOG_PATH = RECORDINGS_24H_CLEAN_DIR / "processing_log.csv"

print(f"EDF source directory: {RECORDINGS_24H_DIR}")
if CLEAN_DIR_EXISTS:
    print(f"Output directory:     {RECORDINGS_24H_CLEAN_DIR} (existing, incremental mode)")
else:
    print(f"Output directory:     {RECORDINGS_24H_CLEAN_DIR} (created, full mode)")
if WRITE_PROCESSING_LOG:
    print(f"Processing log path:  {PROCESSING_LOG_PATH}")
print(f"Default z-score threshold: {DEFAULT_THRESHOLD}")
print(f"Edge zero skip padding: {EDGE_ZERO_SKIP_SEC} s")


## 2 — Discover and group EDF files by modality

In [ ]:
from collections import defaultdict

def parse_edf_filename_from_stem(stem):
    """
    Parse stems like: ECG_2026-02-15_14h or Activity_2026-02-15_14h.
    Returns (modality, date_str) or (None, None) if not matching.
    """
    match = re.match(r'^(Activity|Temperature|[A-Za-z]{3})_(\d{4}-\d{2}-\d{2}[_-]\d+h)$', stem)
    if match:
        return match.group(1), match.group(2)
    return None, None

def date_sort_key(date_str):
    """
    Parse a date_str like '2026-02-15_14h' or '2026-02-15-14h' into a
    (year, month, day, hour) int tuple for chronologically-correct
    comparison/sorting.
    """
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})[_-](\d+)h$', date_str)
    if not m:
        return (date_str,)
    year, month, day, hour = m.groups()
    return (int(year), int(month), int(day), int(hour))

def parse_edf_filename(filepath):
    return parse_edf_filename_from_stem(filepath.stem)


def raw_stem_from_clean_file(filepath):
    stem = filepath.stem
    if stem.endswith("-clean"):
        stem = stem[:-6]
    return stem


def find_last_cleaned_by_modality(output_dir):
    """
    For each modality, find the latest cleaned EDF and map it back to the raw stem.
    """
    last_cleaned = {}
    if not output_dir.exists():
        return last_cleaned

    clean_edfs = sorted(output_dir.glob("*.edf"))
    clean_edfs = [f for f in clean_edfs if f.stem.endswith("-clean")]

    for clean_path in clean_edfs:
        raw_stem = raw_stem_from_clean_file(clean_path)
        modality, date_str = parse_edf_filename_from_stem(raw_stem)
        if modality is None:
            continue

        current = last_cleaned.get(modality)
        if current is None or date_sort_key(date_str) > date_sort_key(current['date_str']):
            last_cleaned[modality] = {
                'date_str': date_str,
                'raw_stem': raw_stem,
                'clean_path': clean_path,
            }

    return last_cleaned


def select_files_to_process(files, last_cleaned_info):
    """
    Resume from the last cleaned file for this modality, inclusive.
    If no clean file exists yet, process the full list.
    """
    if not files:
        return []
    if last_cleaned_info is None:
        return files

    stem_to_idx = {f.stem: idx for idx, f in enumerate(files)}
    raw_stem = last_cleaned_info['raw_stem']
    if raw_stem in stem_to_idx:
        return files[stem_to_idx[raw_stem]:]

    restart_key = date_sort_key(last_cleaned_info['date_str'])
    for idx, f in enumerate(files):
        _, date_str = parse_edf_filename(f)
        if date_str is not None and date_sort_key(date_str) >= restart_key:
            return files[idx:]

    return []              

# Discover all EDF files in the raw directory (exclude derived outputs)
all_edfs = sorted(RECORDINGS_24H_DIR.glob("*.edf"))
all_edfs = [f for f in all_edfs if "-clean" not in f.stem and "-normalized" not in f.stem]

# Group all raw EDFs by modality
modality_groups_all = defaultdict(list)
unmatched = []

for f in all_edfs:
    modality, date_str = parse_edf_filename(f)
    if modality:
        modality_groups_all[modality].append(f)
    else:
        unmatched.append(f)

# last_cleaned_by_modality = {}
# Original line (restore after full rerun):
last_cleaned_by_modality = find_last_cleaned_by_modality(RECORDINGS_24H_CLEAN_DIR) if CLEAN_DIR_EXISTS else {}
modality_groups = {}

print(f"Found {len(all_edfs)} EDF files in {RECORDINGS_24H_DIR}")
print("Modality groups:")
for mod, files in sorted(modality_groups_all.items()):
    files = sorted(files, key=lambda path: date_sort_key(parse_edf_filename_from_stem(path.stem)[1]))
    last_cleaned_info = last_cleaned_by_modality.get(mod)
    selected_files = select_files_to_process(files, last_cleaned_info)
    modality_groups[mod] = selected_files

    if last_cleaned_info is None:
        print(f"  {mod:>10s}: {len(selected_files)} files to process (starting from first file)")
        continue

    clean_name = last_cleaned_info['clean_path'].name
    if selected_files:
        print(
            f"  {mod:>10s}: {len(selected_files)} of {len(files)} files to process "
            f"(resume from {selected_files[0].name}, based on {clean_name})"
        )
    else:
        print(
            f"  {mod:>10s}: 0 files to process "
            f"(no raw file found at or after {clean_name})"
        )

if unmatched:
    print(f"\\n  Unmatched files (skipped): {len(unmatched)}")
    for f in unmatched:
        print(f"    - {f.name}")

## 3 — Helper functions (filter check, artefact detection, saving)

In [ ]:
def check_and_remove_filters(raw):
    """
    Detect if the EDF file has any filtering indicated in its header.
    If filters are found, warn the user and reset the header fields so
    that all downstream processing operates on the raw, unfiltered signal.
    """
    hp = raw.info.get('highpass', 0.0)
    lp = raw.info.get('lowpass', 0.0)
    sfreq = raw.info['sfreq']
    nyquist = sfreq / 2.0
    filters_found = []

    if hp and hp > 0:
        filters_found.append(f'highpass = {hp} Hz')
    if lp and 0 < lp < nyquist:
        filters_found.append(f'lowpass = {lp} Hz')

    if filters_found:
        print(f"  WARNING: Filter(s) detected in EDF header: {', '.join(filters_found)}")
        print(f"    → Resetting header to raw (highpass=0, lowpass={nyquist} Hz)")
        with raw.info._unlock():
            raw.info['highpass'] = 0.0
            raw.info['lowpass'] = nyquist
    else:
        print(f"  No filters in EDF header (raw signal).")

    return raw

def _edge_zero_skip_mask(sig, sfreq, extra_sec=1.0):
    sig = np.asarray(sig)
    edge_mask = np.zeros(sig.shape, dtype=bool)
    finite_nonzero = np.isfinite(sig) & (sig != 0)

    if not finite_nonzero.any():
        edge_mask[:] = True
        return edge_mask

    first_valid = int(np.argmax(finite_nonzero))
    last_valid = int(len(sig) - np.argmax(finite_nonzero[::-1]) - 1)
    extra = int(max(0, round(float(extra_sec) * float(sfreq))))

    lead_end = min(len(sig), first_valid + extra)
    trail_start = max(0, last_valid - extra + 1)

    if lead_end > 0:
        edge_mask[:lead_end] = True
    if trail_start < len(sig):
        edge_mask[trail_start:] = True

    return edge_mask

def _zero_jump_protection_mask(sig):
    is_zero = np.isfinite(sig) & (sig == 0)
    protect = is_zero.copy()
    transitions = np.where(np.diff(is_zero.astype(int)) != 0)[0]
    for idx in transitions:
        protect[max(0, idx)] = True
        protect[min(len(sig) - 1, idx + 1)] = True
    return protect

def _find_nan_gaps(data_1d, fs):
    """Return [[offset_s, duration_s], ...] for every contiguous NaN run."""
    mask = np.isnan(data_1d)
    if not np.any(mask):
        return []
    padded = np.concatenate(([False], mask, [False]))
    diff = np.diff(padded.astype(np.int8))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    return [[float(s / fs), float((e - s) / fs)] for s, e in zip(starts, ends)]

def restore_nan_from_sidecar(data, sfreq, edf_path):
    """
    Restore NaN gaps into a (n_channels, n_samples) array using the .json
    sidecar written by edf_writer_worker.py.  If no sidecar exists, returns
    data unchanged.
    """
    sidecar_path = edf_path.with_suffix(".json")
    if not sidecar_path.exists():
        return data
    with open(sidecar_path) as fh:
        meta = json.load(fh)
    gaps = meta.get("gaps", [])
    if not gaps:
        return data
    result = data.copy()
    n_samples = data.shape[1]
    for offset_s, dur_s in gaps:
        start = int(round(offset_s * sfreq))
        end = min(n_samples, start + int(round(dur_s * sfreq)))
        if start < n_samples:
            result[:, start:end] = np.nan
    print(f"    Restored {len(gaps)} NaN gap(s) from {sidecar_path.name}")
    return result

def detect_artefacts(data, sfreq, threshold=5.0):
    n_channels, n_samples = data.shape
    artefact_mask = np.zeros_like(data, dtype=bool)

    for ch in range(n_channels):
        sig = data[ch]
        protect = _zero_jump_protection_mask(sig)
        edge_skip = _edge_zero_skip_mask(sig, sfreq, extra_sec=EDGE_ZERO_SKIP_SEC)
        # Exclude NaN, zero padding, and edge regions from stats
        valid = (sig != 0) & ~np.isnan(sig) & ~edge_skip
        if valid.sum() < 2:
            continue

        mu = np.mean(sig[valid])
        sigma = np.std(sig[valid])
        if sigma <= 0:
            continue

        z = np.abs((sig - mu) / sigma)
        mask_ch = z > threshold
        mask_ch[protect | edge_skip] = False
        # Never flag NaN positions as artefacts
        mask_ch[np.isnan(sig)] = False
        artefact_mask[ch] = mask_ch

    expand = int(0.1 * sfreq)
    for ch in range(n_channels):
        sig = data[ch]
        protect = _zero_jump_protection_mask(sig)
        edge_skip = _edge_zero_skip_mask(sig, sfreq, extra_sec=EDGE_ZERO_SKIP_SEC)
        mask_ch = artefact_mask[ch].copy()
        indices = np.where(mask_ch)[0]
        for idx in indices:
            start = max(0, idx - expand)
            end = min(n_samples, idx + expand + 1)
            mask_ch[start:end] = True
        mask_ch[protect | edge_skip] = False
        mask_ch[np.isnan(sig)] = False
        artefact_mask[ch] = mask_ch

    total_samples = n_channels * n_samples
    n_artefact = artefact_mask.sum()
    stats = {
        'threshold': threshold,
        'total_samples': total_samples,
        'artefact_samples': int(n_artefact),
        'artefact_pct': 100.0 * n_artefact / total_samples if total_samples > 0 else 0,
        'per_channel': {}
    }
    for ch in range(n_channels):
        ch_art = artefact_mask[ch].sum()
        stats['per_channel'][ch] = {
            'n_artefact': int(ch_art),
            'pct': 100.0 * ch_art / n_samples if n_samples > 0 else 0
        }

    return artefact_mask, stats


def remove_artefacts(data, artefact_mask):
    clean_data = data.copy()
    n_channels = data.shape[0]

    for ch in range(n_channels):
        mask = artefact_mask[ch]
        if not mask.any():
            continue
        # Set artefact samples to NaN — they will be recorded as gaps
        # in the JSON sidecar and can be restored/handled downstream.
        clean_data[ch, mask] = np.nan

    return clean_data

def fmt8(val):
    """Format a value for EDF's 8-character physical range field without stopping execution."""
    for prec in (6, 4, 2, 0):
        s = f"{val:.{prec}f}"
        if len(s) <= 8:
            return float(s), False

    for prec in (2, 1, 0):
        s = f"{val:.{prec}e}"
        if len(s) <= 8:
            return float(s), True

    s = f"{val:.0e}"
    return float(s), True

def save_clean_edf(output_path, raw_original, clean_data):
    n_channels = clean_data.shape[0]
    sfreq = raw_original.info['sfreq']
    ch_names = raw_original.ch_names

    meas_date = raw_original.info.get('meas_date', None)

    # Record NaN gap intervals before overwriting NaN→0 (use first channel;
    # all channels share the same gaps since they come from the same source file)
    gaps = _find_nan_gaps(clean_data[0], sfreq)

    writer = pyedflib.EdfWriter(str(output_path), n_channels,
                                file_type=pyedflib.FILETYPE_EDFPLUS)
    try:
        if meas_date is not None and hasattr(meas_date, 'year'):
            writer.setStartdatetime(meas_date)

        samples_to_write = []
        for ch in range(n_channels):
            # Work on a copy so we don't modify the caller's array
            sig = clean_data[ch].copy()
            np.nan_to_num(sig, copy=False, nan=0.0)

            phys_min = float(np.min(sig))
            phys_max = float(np.max(sig))
            if phys_min == phys_max:
                phys_min -= 1.0
                phys_max += 1.0

            channel_name = (
                ch_names[ch]
                if ch < len(ch_names)
                else f"Ch{ch}"
            )

            if channel_name in calibration.EPHYS_RANGE_UV:
                # fixed range, exact digital values (one digital step = one ADC count)
                phys_min, phys_max = calibration.EPHYS_RANGE_UV[channel_name]
                sig = calibration.to_digital(sig, phys_min, phys_max)
                trunc_min = trunc_max = False
            else:
                phys_min, trunc_min = fmt8(phys_min)
                phys_max, trunc_max = fmt8(phys_max)

            if trunc_min or trunc_max:
                print(
                    f"  WARNING: physical range for channel "
                    f"{channel_name} was stored with reduced precision."
                )

            ch_dict = {
                'label': channel_name,
                'dimension': calibration.OUTPUT_DIMENSION[channel_name],
                'sample_frequency': sfreq,
                'physical_min': phys_min,
                'physical_max': phys_max,
                'digital_min': -32768,
                'digital_max': 32767,
                'transducer': calibration.TEMPERATURE_TRANSDUCER if channel_name == 'Temperature' else '',
                'prefilter': '',
            }
            writer.setSignalHeader(ch, ch_dict)
            samples_to_write.append(sig)

        writer.writeSamples(samples_to_write, digital=ch_names[0] in calibration.EPHYS_RANGE_UV)
    finally:
        writer.close()

    # Write JSON sidecar (same format as edf_writer_worker.py)
    sidecar_path = output_path.with_suffix(".json")
    starttime_unix = meas_date.timestamp() if meas_date is not None else 0.0
    sidecar_payload = {
        "starttime_unix": float(starttime_unix),
        "fs": float(sfreq),
        "n_samples": int(clean_data.shape[1]),
        "label": ch_names[0] if ch_names else "Ch0",
        "gaps": gaps,
    }
    # keep the extra keys (Temperature calibration) of the input sidecar
    input_sidecar = Path(raw_original.filenames[0]).with_suffix(".json")
    if input_sidecar.exists():
        with open(input_sidecar) as fh:
            sidecar_payload.update({k: v for k, v in json.load(fh).items() if k not in sidecar_payload})
    with open(sidecar_path, "w") as fh:
        json.dump(sidecar_payload, fh)

    gap_msg = f" ({len(gaps)} gap(s) preserved)" if gaps else " (no gaps)"
    print(f" Saved: {output_path.name} + {sidecar_path.name}{gap_msg}")

## 4 — Plotting function (matplotlib, non-interactive, dual axes)

In [ ]:
def _axis_limits_ignore_zero_jumps(signal, pad_ratio=0.08):
    sig = np.asarray(signal)
    valid = np.isfinite(sig) & (sig != 0)
    vals = sig[valid]

    if vals.size == 0:
        vals = sig[np.isfinite(sig)]
    if vals.size == 0:
        return (-1.0, 1.0)

    ymin = float(np.min(vals))
    ymax = float(np.max(vals))
    if ymin == ymax:
        spread = max(1e-6, abs(ymin) * 0.1)
        return ymin - spread, ymax + spread

    pad = (ymax - ymin) * pad_ratio
    return ymin - pad, ymax + pad


def _plot_nonzero_only(signal):
    sig = np.asarray(signal, dtype=float).copy()
    sig[(np.isfinite(sig)) & (sig == 0)] = np.nan
    return sig


def _visible_x_limits(time_axis, *signals):
    time_axis = np.asarray(time_axis)
    visible = np.zeros(time_axis.shape, dtype=bool)
    for sig in signals:
        arr = np.asarray(sig)
        visible |= np.isfinite(arr) & (arr != 0)

    idx = np.where(visible)[0]
    if idx.size == 0:
        return None
    return float(time_axis[idx[0]]), float(time_axis[idx[-1]])


def plot_file_dual_axes(raw_data, clean_data, artefact_mask, raw, file_idx,
                        filename, threshold, stats, max_points=50_000):
    n_channels, n_samples = raw_data.shape
    ds = max(1, n_samples // max_points)
    times = raw.times[::ds]

    if times[-1] > 300:
        time_axis = times / 60.0
        time_label = "Time (minutes)"
    else:
        time_axis = times
        time_label = "Time (seconds)"

    fig, axes = plt.subplots(n_channels, 1, figsize=(16, 3 * n_channels + 1),
                             sharex=True, squeeze=False)
    axes = axes.flatten()

    fig.suptitle(
        f"File #{file_idx}  —  {filename}\n"
        f"Threshold = {threshold} SD  —  {stats['artefact_pct']:.2f}% flagged overall",
        fontsize=13, fontweight='bold', y=1.0
    )

    for ch in range(n_channels):
        ax_left = axes[ch]
        ax_right = ax_left.twinx()

        sig_raw = raw_data[ch, ::ds]
        sig_clean = clean_data[ch, ::ds]
        mask_ds = artefact_mask[ch, ::ds]

        sig_raw_plot = _plot_nonzero_only(sig_raw)
        sig_clean_plot = _plot_nonzero_only(sig_clean)

        sig_raw_clean_part = sig_raw_plot.copy()
        sig_raw_clean_part[mask_ds] = np.nan
        sig_raw_art_part = np.full_like(sig_raw_plot, np.nan)
        sig_raw_art_part[mask_ds] = sig_raw_plot[mask_ds]

        ax_right.plot(time_axis, sig_raw_clean_part, color='silver', linewidth=0.4,
                      alpha=0.6, label='Raw')
        ax_right.plot(time_axis, sig_raw_art_part, color='red', linewidth=0.8,
                      alpha=0.9, label='Artefact')
        ax_right.set_ylabel('Raw', color='grey', fontsize=9)
        ax_right.tick_params(axis='y', labelcolor='grey', labelsize=8)

        ax_left.plot(time_axis, sig_clean_plot, color='steelblue', linewidth=0.5,
                     alpha=0.9, label='Clean')
        ax_left.set_ylabel('Clean', color='steelblue', fontsize=9)
        ax_left.tick_params(axis='y', labelcolor='steelblue', labelsize=8)

        y_left = _axis_limits_ignore_zero_jumps(sig_clean_plot)
        y_right = _axis_limits_ignore_zero_jumps(np.concatenate([sig_raw_clean_part, sig_raw_art_part]))
        ax_left.set_ylim(*y_left)
        ax_right.set_ylim(*y_right)

        x_limits = _visible_x_limits(time_axis, sig_raw_clean_part, sig_raw_art_part, sig_clean_plot)
        if x_limits is not None:
            ax_left.set_xlim(*x_limits)

        ch_name = raw.ch_names[ch]
        ch_stats = stats['per_channel'][ch]
        ax_left.set_title(f"{ch_name}  —  {ch_stats['pct']:.2f}% artefacts",
                          fontsize=10, loc='left')

        ax_left.set_zorder(ax_right.get_zorder() + 1)
        ax_left.patch.set_visible(False)

    axes[-1].set_xlabel(time_label, fontsize=10)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='steelblue', linewidth=1.5, label='Clean (left axis)'),
        Line2D([0], [0], color='silver', linewidth=1.5, label='Raw (right axis)'),
        Line2D([0], [0], color='red', linewidth=1.5, label='Artefact (right axis)'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3,
              fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    from IPython.display import display
    display(fig)
    plt.close(fig)

## 6 - Main processing loop

For each modality:

1. Resume from the last cleaned EDF if `OUTPUT_ROOT/recordings-24h-clean/<RAT_ID>` exists; otherwise, start from the first file.
2. Process files in chunks of up to `CHUNK_SIZE = 14` to limit memory usage.
3. For each chunk:

   * load the EDF files;
   * detect artefacts using the current threshold;
   * display the raw and cleaned signals;
   * allow global or per-file threshold adjustment;
   * save the cleaned EDFs;
   * release the chunk from memory before loading the next one.

In [ ]:
# ── Maximum number of files processed at once to limit memory usage ──
CHUNK_SIZE = 14

import gc
import sys
import csv
import shutil

_PROCESSING_LOG_HEADER = [
    "recording_start", "modality", "raw_filename", "clean_filename",
    "threshold", "n_artefact_samples", "artefact_pct",
]

def _append_processing_log(modality, entry, clean_filename):
    """Append one processing record to processing_log.csv."""
    file_exists = PROCESSING_LOG_PATH.exists()

    meas_date = entry["raw"].info.get("meas_date")
    recording_start = (
        meas_date.isoformat(timespec="seconds")
        if meas_date is not None and hasattr(meas_date, "isoformat")
        else ""
    )

    with open(PROCESSING_LOG_PATH, "a", newline="") as fh:
        writer = csv.writer(fh)

        if not file_exists:
            writer.writerow(_PROCESSING_LOG_HEADER)

        writer.writerow([
            recording_start,
            modality,
            entry["path"].name,
            clean_filename,
            entry["threshold"],
            entry["stats"]["artefact_samples"],
            f"{entry['stats']['artefact_pct']:.4f}",
        ])
    
MODALITIES_TO_PROCESS = None  #<--CHANGE HERE  # set to None to process every modality again

for modality in sorted(modality_groups.keys()):
    if MODALITIES_TO_PROCESS and modality not in MODALITIES_TO_PROCESS:
        continue

    # ── Close all open figures from previous modality to free memory ──
    plt.close('all')

    files = modality_groups[modality]
    print(f"\n{'='*80}")
    print(f"  MODALITY: {modality} — {len(files)} file(s) selected")
    print(f"{'='*80}")

    if not files:
        print("  Nothing to process for this modality.")
        continue

    # ── Split this modality into chunks of at most CHUNK_SIZE files ──
    n_chunks = (len(files) + CHUNK_SIZE - 1) // CHUNK_SIZE
    print(f"  Splitting into {n_chunks} chunk(s) of up to {CHUNK_SIZE} file(s) each.")

    global_threshold = DEFAULT_THRESHOLD

    for chunk_idx in range(n_chunks):
        chunk_start = chunk_idx * CHUNK_SIZE
        chunk_end = min(chunk_start + CHUNK_SIZE, len(files))
        chunk_files = files[chunk_start:chunk_end]

        print(f"\n  {'-'*76}")
        print(f"  CHUNK {chunk_idx + 1}/{n_chunks} — files {chunk_start}..{chunk_end - 1} "
              f"({len(chunk_files)} file(s))")
        print(f"  {'-'*76}")

        # Free memory left over from the previous chunk
        plt.close('all')
        gc.collect()

        # ── Load all files of the current chunk ──
        file_data = []  # list of dicts: {path, raw, data, threshold, artefact_mask, stats, clean_data}

        for local_idx, edf_path in enumerate(chunk_files):
            # Keep the global file index across chunks for clear logging/refinement
            idx = chunk_start + local_idx

            output_name = edf_path.stem + "-clean.edf"
            output_path = RECORDINGS_24H_CLEAN_DIR / output_name

            if output_path.exists():
                print(f"\n  [File #{idx}] Reloading for overwrite: {edf_path.name}")
            else:
                print(f"\n  [File #{idx}] Loading: {edf_path.name}")

            raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose=False)
            raw = check_and_remove_filters(raw)
            # MNE returns volts for uV headers; clean_scale gives uV back (Activity: x 1e-6, intentional)
            data = raw.get_data() * calibration.clean_scale(raw.ch_names[0])

            # Restore NaN at positions flagged as missing in the .json sidecar
            data = restore_nan_from_sidecar(data, raw.info['sfreq'], edf_path)

            print(f"    Channels: {raw.ch_names}")
            print(f"    Sampling rate: {raw.info['sfreq']} Hz")
            print(f"    Duration: {raw.times[-1]:.1f} s ({raw.times[-1]/3600:.2f} h)")

            file_data.append({
                'idx': idx,
                'path': edf_path,
                'raw': raw,
                'data': data,
                'threshold': global_threshold,
                'artefact_mask': None,
                'stats': None,
                'clean_data': None,
            })

        if not file_data:
            print("  Nothing to process in this chunk.")
            continue

        # ── Function to (re-)process one entry ──
        def process_entry(entry):
            mask, stats = detect_artefacts(entry['data'], entry['raw'].info['sfreq'],
                                           threshold=entry['threshold'])
            entry['artefact_mask'] = mask
            entry['stats'] = stats
            entry['clean_data'] = remove_artefacts(entry['data'], mask)

        # ── Function to process ALL entries of the chunk ──
        def process_all(entries, threshold):
            for e in entries:
                e['threshold'] = threshold
                process_entry(e)

        # ── Function to display ALL entries of the chunk ──
        def display_all(entries):
            for e in entries:
                print(f"\n  --- File #{e['idx']}: {e['path'].name}  "
                      f"(threshold={e['threshold']}) ---")
                for ch_idx, ch_name in enumerate(e['raw'].ch_names):
                    ch_stats = e['stats']['per_channel'][ch_idx]
                    print(f"      {ch_name}: {ch_stats['n_artefact']} samples "
                          f"({ch_stats['pct']:.2f}%)")
                print(f"      TOTAL: {e['stats']['artefact_samples']} samples "
                      f"({e['stats']['artefact_pct']:.2f}%)")

                plot_file_dual_axes(
                    e['data'], e['clean_data'], e['artefact_mask'],
                    e['raw'], e['idx'], e['path'].name,
                    e['threshold'], e['stats']
                )

        # ── Initial processing of this chunk ──
        print(f"\n  Processing chunk {chunk_idx + 1}/{n_chunks} with threshold = {global_threshold} SD ...")
        process_all(file_data, global_threshold)
        display_all(file_data)

        # ── Interactive refinement loop (scoped to this chunk) ──
        while True:
            sys.stdout.flush()
            resp = input(
                f"\n  Change threshold GLOBALLY for the {len(file_data)} file(s) "
                f"of chunk {chunk_idx + 1}/{n_chunks} ({modality})? [y/n]: "
            ).strip().lower()

            if resp in ('y', 'yes'):
                sys.stdout.flush()
                new_thresh = input("  Enter new global threshold (z-score, e.g. 3.0): ").strip()
                try:
                    global_threshold = float(new_thresh)
                except ValueError:
                    print(f"  → Invalid input '{new_thresh}', keeping current thresholds.")
                    continue
                print(f"  → Re-processing the {len(file_data)} file(s) of this chunk with threshold = {global_threshold} ...")
                process_all(file_data, global_threshold)
                display_all(file_data)
                continue

            # Not global — ask about individual file
            sys.stdout.flush()
            resp2 = input(
                "  Change threshold for a SPECIFIC file in this chunk? [y/n]: "
            ).strip().lower()

            if resp2 not in ('y', 'yes'):
                break

            # Per-file refinement loop
            while True:
                sys.stdout.flush()
                file_num = input("  Enter file # to adjust: ").strip()
                try:
                    file_num = int(file_num)
                except ValueError:
                    print(f"  → Invalid file number '{file_num}'.")
                    continue

                target = None
                for e in file_data:
                    if e['idx'] == file_num:
                        target = e
                        break

                if target is None:
                    valid_ids = [str(e['idx']) for e in file_data]
                    print(f"  → File #{file_num} not in current chunk. "
                          f"Valid file numbers for this chunk: {', '.join(valid_ids)}")
                    continue

                sys.stdout.flush()
                new_thresh = input(
                    f"  Enter new threshold for file #{file_num} (current={target['threshold']}): "
                ).strip()
                try:
                    target['threshold'] = float(new_thresh)
                except ValueError:
                    print(f"  → Invalid threshold '{new_thresh}', keeping {target['threshold']}.")
                    continue

                print(f"  → Re-processing file #{file_num} with threshold = {target['threshold']} ...")
                process_entry(target)

                print(f"\n  --- File #{target['idx']}: {target['path'].name}  "
                      f"(threshold={target['threshold']}) ---")
                for ch_idx, ch_name in enumerate(target['raw'].ch_names):
                    ch_stats = target['stats']['per_channel'][ch_idx]
                    print(f"      {ch_name}: {ch_stats['n_artefact']} samples "
                          f"({ch_stats['pct']:.4f}%)")
                print(f"      TOTAL: {target['stats']['artefact_samples']} samples "
                      f"({target['stats']['artefact_pct']:.4f}%)")

                plot_file_dual_axes(
                    target['data'], target['clean_data'], target['artefact_mask'],
                    target['raw'], target['idx'], target['path'].name,
                    target['threshold'], target['stats']
                )

                sys.stdout.flush()
                more = input("  Adjust another file? [y/n]: ").strip().lower()
                if more not in ('y', 'yes'):
                    break

            print("\n  Current state of all files in this chunk:")
            display_all(file_data)

        # ── Save all files of this chunk ──
        print(f"\n  Saving the {len(file_data)} file(s) of chunk {chunk_idx + 1}/{n_chunks} ...")
        for e in file_data:
            output_name = e['path'].stem + "-clean.edf"
            output_path = RECORDINGS_24H_CLEAN_DIR / output_name
            action = "Overwriting" if output_path.exists() else "Saving"
            print(f"  {action}: {output_name}")
            try:
                save_clean_edf(output_path, e['raw'], e['clean_data'])
                if WRITE_PROCESSING_LOG:
                    _append_processing_log(modality, e, output_name)
                print(f" Done: {e['path'].name} → {output_name}")
            except Exception as ex:
                print(f" FAILED to save {e['path'].name} → {output_name}: {ex}")
            continue
        print(f" Done: {e['path'].name} → {output_name}")

        # ── Free memory before loading the next chunk ──
        print(f"\n  Releasing memory used by chunk {chunk_idx + 1}/{n_chunks} ...")
        for e in file_data:
            e['raw'] = None
            e['data'] = None
            e['clean_data'] = None
            e['artefact_mask'] = None
            e['stats'] = None
        file_data.clear()
        del file_data
        plt.close('all')
        gc.collect()

print(f"\n{'='*80}")
print("  ALL DONE")
print(f"{'='*80}")